# model_test — 팀 실험 예시

새 특징(H2·H4 등)을 넣거나 설정을 바꿔 실험하는 **예시**입니다. 이 노트북을 복사해 고쳐 써도 되고, 새 노트북이나 새 py를 만들어도 됩니다.
결과를 서로 비교하려면 아래 공통 함수만 쓰면 됩니다.

| 무엇 | 함수 | 지키는 규칙 |
|---|---|---|
| 입력 준비 | `mt.prepare_inputs(ROOT, config, extra_features=...)` | 입력은 `data/`와 config만. 새 특징은 매달 순위 [−1, 1], 학습은 쓰는 특징이 모두 관측된 행만 |
| Fold | `mt.plan_folds`, `mt.fold_rows` | 2003년부터 확장창, 직전 1년으로 고르고 다시 학습하지 않음 |
| 채점 행 | `mt.validation_rows` | 2006~2021년 수익률 있는 전 종목 133,876행, 빈 특징은 0 |
| 저장 | `mt.finalize_boosting` / `mt.finalize_linear` / `mt.save_result` | 종료 검사 통과한 것만 표 저장. `author`·`experiment`를 주면 공유용 폴더 이름 |

- **공유**: 코드·노트북·결과는 Git, 입력 `data/`만 Drive. `data/`는 Drive에서 내려받아 저장소 안 `data/`에 그대로 둡니다(`data/interim/panel.parquet` 등 경로가 같아야 함).
- **결과는 `outputs/model_results/<작성자>/<시각>_<실험>_<모델·단계>/`에 같은 규격으로 생기고, 그 폴더를 Git에 커밋해 공유**합니다. 예측값 파일(`validation_predictions.parquet`)은 커서 Git에서 빠지고 각자 PC에만 남습니다(비교에는 쓰지 않음). `outputs/run_id`는 개인 기록이고 공유하지 않습니다.
- XGBoost는 GPU(Colab)로 돌립니다. Colab 결과는 내 드라이브에 저장 → 내려받아 로컬에 넣고 커밋합니다.
- **이 노트북은 고치지 말고 `notebook/<내 이름>/`에 복사해서** 씁니다. 공용 파일(이 노트북, `src/`의 기존 모듈, `config/experiment.yaml`, `requirements.txt`)을 고치면 Git 충돌이 납니다. 내 코드는 `src/team/<내 이름>/`에 새로 만듭니다.
- 실험 내용(작성자, 실험 이름, 특징 파일, 설정, 입력 조합, 바꾼 값)은 **1·3번 셀의 변수**로 적습니다. 저장할 때 run.json에 자동으로 기록됩니다.
- 2022~2025년(테스트 기간)은 읽지 않습니다. `data/raw/`의 `test_2022_2025__*.csv`를 열거나 폴더째 검색·복사하지 않습니다.

## 0. 실행 환경 확인

In [ ]:
import os, sys, platform, subprocess, importlib.util, shutil
from pathlib import Path

print('python      :', sys.version.split()[0], platform.platform())
print('작업 폴더    :', os.getcwd())
print('/content 존재:', Path('/content').exists())

# 저장소가 보이는지: 위로 올라가며 src/와 config/experiment.yaml을 찾는다
found = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file()), None)
print('저장소 루트   :', found)

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)
else:
    print('GPU          : 없음')

for m in ['numpy', 'pandas', 'pyarrow', 'scipy', 'sklearn', 'xgboost', 'lightgbm', 'yaml']:
    if importlib.util.find_spec(m):
        mod = __import__(m)
        print(f'{m:9s}', getattr(mod, '__version__', '?'))
    else:
        print(f'{m:9s}', '설치 안 됨')

## 1. 누가·무엇을·어디에

- `AUTHOR`, `EXPERIMENT`는 결과 폴더 이름에 들어갑니다(영문·숫자·`_`·`-`만, 한글 불가). 기본 실험이면 `EXPERIMENT = 'base'`.
- `FEATURES`: 새 특징 파일. 형식은 2번 셀 설명.
- **로컬**: 결과는 `outputs/model_results/<AUTHOR>/` 아래에 생깁니다.
- **Colab**: `GIT_REF`에 **브랜치 이름**(예: `main`, `hong`)을 적는다. 매번 GitHub에서 그 브랜치의 최신 코드를 받으므로, 로컬에서 고친 것은 **푸시한 뒤** 돌린다. 받은 커밋은 run.json에 기록되고, 같은 결과를 다시 만들 때만 그 커밋 해시를 넣는다. `data/`는 내 드라이브의 `금융데이터인텔리전스`(공유 폴더 바로가기)에서 연결, 결과는 내 드라이브 `model_results/<AUTHOR>/`.

In [ ]:
import os, sys, re, shutil, subprocess, time
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

AUTHOR = ''                # 예: 'hong' (결과 폴더 outputs/model_results/<AUTHOR>/)
EXPERIMENT = 'base'        # 예: 'h2_depth5' (결과 폴더 이름과 run.json에 기록)
DESCRIPTION = ''           # 한 줄 설명(run.json에 기록)
FEATURES = {}              # 새 특징 파일, 예: {'h2': 'data/processed/h2.parquet'}
GIT_REF = ''              # Colab에서만: 브랜치 이름(예: 'main', 'hong') — 그 브랜치의 GitHub 최신 코드. 특정 시점 재현은 커밋 해시

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')
if not AUTHOR:
    raise ValueError('AUTHOR를 채운다(결과 폴더 이름에 쓰임).')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise RuntimeError("GIT_REF에 브랜치 이름(예: 'main', 'hong')을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    # GitHub에서 최신을 받아 GIT_REF로 맞춘다: 브랜치 이름이면 원격 최신(origin/<브랜치>), 아니면 커밋 해시
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    print('받은 코드:', GIT_REF, '@', subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    if not (ROOT / 'data').is_symlink():
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = Path('/content/drive/MyDrive/model_results')   # Colab: 내 드라이브(개인)에 같은 구조로 → 내려받아 로컬 outputs/model_results/에 넣고 커밋
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None                                   # 로컬: outputs/model_results

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.evaluation import model_test as mt
from src.evaluation import model_summary as ms

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_grid(config)
pins = dict(re.findall(r'^([A-Za-z0-9_.-]+)==([^\s#]+)', (ROOT / 'requirements.txt').read_text(encoding='utf-8'), re.M))
for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'pyarrow', 'PyYAML', 'xgboost', 'lightgbm']:
    try:
        got = version(name)
    except PackageNotFoundError:
        got = '없음'
    print(f"{name:13s} 고정 {pins.get(name, '?'):8s} 설치 {got:8s} {'' if got == pins.get(name) else '← 다름'}")
GPU_NAME = (subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
            if shutil.which('nvidia-smi') else None)
N_THREADS = os.cpu_count()
started = datetime.now()
pd.set_option('display.width', 200, 'display.max_rows', 200, 'display.float_format', '{:.4f}'.format)
print('ROOT  :', ROOT)
print('저장  :', (OUT_ROOT or ROOT / config['PATHS']['model_results']) / AUTHOR)
print('GPU   :', GPU_NAME, '| 스레드:', N_THREADS, '| 작성자:', AUTHOR, '| 실험:', EXPERIMENT)

## 2. 입력 준비 (약 2분)

- `FEATURES`: 새 특징 파일 `{이름: 'data/processed/<파일>.parquet'}`. 열은 `stock_id`, `base_month`(YYYY-MM), `<이름>`(없으면 NaN).
  종목 × 기준월 한 줄, 기준월 2000-12~2021-11만. **t월 말에 실제로 알 수 있던 값**이어야 합니다(만든 사람 책임).
- 새 특징을 넣으면 학습 행이 그 특징까지 관측된 행으로 줄어듭니다. 비교하려면 **기존 입력도 같은 실험 안에서 함께** 돌립니다.
- 확인할 숫자: 채점 행 **133,876**(항상 같음), σ 일치 검사 통과. 새 특징이 없으면 학습 행 **151,761**.

In [ ]:
t0 = time.perf_counter()
prepared = mt.prepare_inputs(ROOT, config, extra_features=FEATURES or None)
print(f'입력 준비 {time.perf_counter() - t0:.0f}초')
for k, v in prepared['checks'].items():
    print(f'  {k:44s} {v}')
print('쓸 수 있는 특징:', mt.feature_names(prepared))
prepared['feature_coverage'].head(20) if len(prepared['feature_coverage']) else None

## 3. 예시 A — 기존 부스팅 실행기로 설정 바꿔 돌리기

- `custom=True`면 승인된 값 밖의 깊이·학습률도 받습니다. `OVERRIDES`로 라이브러리 설정(규제·추출 비율 등)을 바꿉니다.
  목적함수·장치·트리 수(1000)는 바꿀 수 없습니다. 추출 비율을 1보다 작게 하면 난수 시드(`seed`, 기본 0)가 결과를 정합니다.
- 장치: XGBoost는 GPU(Colab), LightGBM은 CPU. 같은 실험을 다른 장치로 돌리면 결과가 달라집니다(run.json에 기록).
- 설정 수 × 입력 조합 수 × Fold 16 × (σ가 있으면 λ 3개)만큼 학습합니다. LightGBM 설정 1개·조합 1개가 로컬 약 6분입니다.

In [ ]:
KIND = 'lgb_rankreg'
SETTINGS = [(KIND, 'percentile', 'mse', d, 0.01) for d in (3,)]           # (종류, 목표값, 손실, 깊이, 학습률)
INPUTS = [['size', 'bm', 'mom', 'sigma']]                                  # 새 특징이 있으면 예: ['size', 'bm', 'mom', 'sigma', 'h2']
OVERRIDES = {}                                                             # 예: {'lambda_l2': 5.0}
SETS = dict(mt.make_input_set(c) for c in INPUTS)

t0 = time.perf_counter()
result = mt.run_boosting(prepared, config, KIND, phase='individual', settings=SETTINGS, sets=SETS, custom=True,
                         overrides=OVERRIDES, n_threads=N_THREADS)
print(f'계산 {(time.perf_counter() - t0) / 60:.1f}분')
saved = mt.finalize_boosting(ROOT, config, prepared, result, started=started, out_root=OUT_ROOT, gpu_name=GPU_NAME,
                             author=AUTHOR, experiment=EXPERIMENT, description=DESCRIPTION,
                             changes={'settings': SETTINGS, 'inputs': INPUTS, 'overrides': OVERRIDES, 'features': FEATURES})
print(saved['status'], saved['run_dir'])
mt.boosting_part_summary(result)

## 4. 예시 B — 직접 만든 모델을 같은 형식으로 저장하기

모델은 자유입니다. 지킬 것은 세 가지입니다.
1. Fold마다 `mt.fold_rows`의 `fit` 행으로만 학습하고(설정을 고른다면 `inner` 행으로), `out` 행을 예측합니다.
2. 예측의 빈 특징은 0으로 채웁니다(학습 행에는 빈 값이 없습니다).
3. `mt.save_result`에 예측(채점 행 순서)과 Fold별 선택 표를 넘깁니다.

아래는 튜닝 없는 OLS(scikit-learn) 예시입니다. 입력 열은 순위 [−1, 1]: `x_size`, `x_bm`, `x_mom`, `x_sigma_<span>`, 새 특징은 `x_<이름>`.

In [ ]:
from sklearn.linear_model import LinearRegression

COLS = ['x_size', 'x_bm', 'x_mom', 'x_sigma_span36']
TARGET = 't_monthly_zscore'                       # 목표값: t_raw, t_winsor_1_99, t_monthly_zscore, t_percentile, t_decile
data = prepared['data']
val = mt.validation_rows(prepared, config)
pred = np.full(len(val), np.nan)
rows = []
for f in mt.plan_folds(config['MODEL_GRID']):
    m = mt.fold_rows(prepared, f)
    X, y = data.loc[m['fit'], COLS].to_numpy(float), data.loc[m['fit'], TARGET].to_numpy(float)
    model = LinearRegression().fit(X, y)
    pred[val.fold.eq(f['fold']).to_numpy()] = model.predict(data.loc[m['out'], COLS].fillna(0).to_numpy(float))
    train_ic = mt.mean_ic(mt.monthly_ic(data.target_month.to_numpy()[m['fit']], model.predict(X), data.y.to_numpy(float)[m['fit']]))
    rows.append({'target': 'monthly_zscore', 'loss': 'mse', 'input_set': 'ind:size+bm+mom+sigma', 'hparams': 'span36',
                 'fold': f['fold'], 'valid_year': f['valid_year'], 'train_ic': train_ic,
                 **{'coef_' + c: w for c, w in zip(COLS, model.coef_)}})
saved_b = mt.save_result(ROOT, config, prepared, model='ols_sklearn', started=started, author=AUTHOR, experiment=EXPERIMENT,
                         description=DESCRIPTION, out_root=OUT_ROOT,
                         predictions={('monthly_zscore', 'mse', 'ind:size+bm+mom+sigma', 'span36'): pred},
                         selections=pd.DataFrame(rows), changes={'cols': COLS, 'target': TARGET})
print(saved_b['status'], saved_b['run_dir'])
saved_b['tables']['performance_summary']

## 5. 저장한 결과가 비교 표에 들어가는지 확인

내 결과 폴더(`<model_results>/<AUTHOR>/`) 중 PASSED만 모아 합칩니다. 합치기 검사(입력 해시·평가 기준)에 걸리면 이유가 나옵니다.

In [ ]:
loaded = ms.load_runs(ms.find_run_folders((OUT_ROOT or ROOT / config['PATHS']['model_results']) / AUTHOR))
display(loaded['runs'][['run', 'author', 'experiment', 'train_rows_2003_2021', 'family', 'kind', 'device']])
display(ms.all_settings(loaded, experiment=EXPERIMENT, sets=None).head(20))